# NB07 — Robustness (source & script held-out)

Generalization stress test. For each held-out config the 3 encoders are trained **from scratch** on the
held-out train (never having seen the held-out source/script), fused by a val-optimised weighted
ensemble, and evaluated on the unseen held-out test.

- **Source-held-out** ×4: test = one unseen source.
- **Script-held-out** ×2: test = one unseen script.
- **Hard leakage guard** per config: `train∪val ∩ test == 0` on `uid`.

Same method as NB05 (FT+FGM), uniform LR. 1 seed per encoder to keep
it tractable (18 runs). Set `SEEDS`/toggles in the config cell to trade off cost vs. completeness.


In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"   # safe even with num_workers=0; prevents warnings
import torch
torch.set_num_threads(4)   # keep 4 CPU threads for torch ops; rest are free for OS

In [2]:
import os, json, glob, time, math, random, warnings
from collections import defaultdict
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
from scipy.optimize import minimize
from sklearn.metrics import f1_score, accuracy_score, matthews_corrcoef, roc_auc_score, classification_report
warnings.filterwarnings("ignore")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("CUDA:", torch.cuda.is_available())

CUDA: True


In [ ]:
# ── CONFIG (mirrors NB05; lighter seed budget) ──────────────────────────────
DEBUG = False; DEBUG_SAMPLES = 400; DEBUG_EPOCHS = 2
CFG = {"models": {
       "banglishbert": "csebuetnlp/banglishbert",
       "banglabert": "csebuetnlp/banglabert",
       "muril": "google/muril-base-cased",
       "xlmr": "xlm-roberta-base"},
       "max_length":128,"batch_size":32,"eval_batch_size":128,"grad_accum_steps":1,"num_workers":0,
       "epochs":8,"encoder_lr":2e-5,"head_lr":8e-5,"weight_decay":0.01,"warmup_ratio":0.10,
       "max_grad_norm":1.0,"label_smoothing":0.03,"focal_gamma":2.0,"class_weight_beta":0.999,
       "dropout":0.25,"head_hidden_dim":384,"n_msd":1,"patience":2,"use_fp16":torch.cuda.is_available(),
       "use_focal":False,"use_cw":False,
       "use_balanced_sampler":False,"sampler_alpha":0.5,
       "use_fgm":True,"fgm_epsilon":1.0,"use_rdrop":False,"rdrop_alpha":0.5,"use_ema":False,"ema_decay":0.999,
       "banglabert_script_only": True,
       "banglabert_script_key": "banglabert"}
SEEDS = [42]                       # robustness: 1 seed/encoder (set [42,123] for variance)
RUN_MODELS = ["banglishbert", "banglabert", "muril", "xlmr"]
SPLIT_DIR = "../data/splits"; OUT = "../outputs/robustness"; os.makedirs(OUT, exist_ok=True)
LABEL, TEXT = "label5", "text_clean"; FORCE = False
if DEBUG: CFG["epochs"] = DEBUG_EPOCHS; print("⚠ DEBUG")
# discover held-out configs
configs = sorted({os.path.basename(p).rsplit("_",1)[0]
                  for p in glob.glob(f"{SPLIT_DIR}/source_holdout_*_train.csv") +
                           glob.glob(f"{SPLIT_DIR}/script_holdout_*_train.csv")})
print("held-out configs:", configs)

held-out configs: ['script_holdout_bangla', 'script_holdout_romanized', 'source_holdout_banth', 'source_holdout_bd_shs', 'source_holdout_facebook_44001', 'source_holdout_multilabel_12557']


In [4]:
# ── Model / loss / sampler / FGM / EMA (same as NB05) ───────────────────────
class DS(Dataset):
    def __init__(self, df, tok, maxlen, enc):
        self.texts=df.reset_index(drop=True)[TEXT].fillna("").astype(str).tolist()
        self.labels=[int(enc.get(v,-1)) for v in df[LABEL]]; self.tok,self.maxlen=tok,maxlen
    def __len__(self): return len(self.texts)
    def __getitem__(self,i):
        e=self.tok(self.texts[i],max_length=self.maxlen,truncation=True,padding=False)
        it={k:torch.tensor(v,dtype=torch.long) for k,v in e.items()}; it["label"]=torch.tensor(self.labels[i],dtype=torch.long); return it
class Collator:
    def __init__(self,tok): self.tok=tok
    def __call__(self,fs):
        labels=torch.stack([f["label"] for f in fs]); feats=[{k:v for k,v in f.items() if k!="label"} for f in fs]
        b=self.tok.pad(feats,padding=True,return_tensors="pt"); b["label"]=labels; return b
class FocalLoss(nn.Module):
    def __init__(self,gamma=2.0,weight=None,ls=0.0): super().__init__(); self.g,self.w,self.ls=gamma,weight,ls
    def forward(self,lg,t):
        ce=F.cross_entropy(lg,t,weight=self.w,reduction="none",label_smoothing=self.ls); return (((1-torch.exp(-ce))**self.g)*ce).mean()
def build_cw(series,enc,beta=0.999,max_w=10.0):
    m=series.map(enc).dropna().astype(int); n=len(enc); c=m.value_counts().sort_index(); w=np.ones(n,dtype=np.float32)
    for i in range(n):
        k=int(c.get(i,0))
        if k>0: w[i]=1.0/max((1.0-beta**k)/max(1.0-beta,1e-12),1e-9)
    w=np.clip(w,w.min(),w.min()*max_w); w=w/w.mean(); return torch.tensor(w,dtype=torch.float32,device=device)
def make_sampler(df,enc,alpha=0.5):
    y=df[LABEL].map(enc).fillna(-1).astype(int).values; cc=np.bincount(y[y>=0],minlength=len(enc)).astype(float); cc[cc==0]=1.0
    cw=(1.0/cc)**float(alpha); sw=np.where(y>=0,cw[np.clip(y,0,None)],0.0)
    return WeightedRandomSampler(torch.as_tensor(sw,dtype=torch.double),len(sw),replacement=True)
class MSDHead(nn.Module):
    def __init__(self,h,n,dropout=0.25,inner=384,n_msd=4):
        super().__init__(); inner=min(inner,h); self.pre=nn.Sequential(nn.Linear(h,inner),nn.GELU(),nn.LayerNorm(inner))
        self.drops=nn.ModuleList([nn.Dropout(dropout) for _ in range(max(1,n_msd))]); self.out=nn.Linear(inner,n)
    def forward(self,x):
        h=self.pre(x)
        if self.training and len(self.drops)>1: return torch.stack([self.out(d(h)) for d in self.drops],0).mean(0)
        return self.out(self.drops[0](h))
class Encoder(nn.Module):
    def __init__(self,name,n,dropout=0.25,inner=384,n_msd=4):
        super().__init__(); self.encoder=AutoModel.from_pretrained(name); h=self.encoder.config.hidden_size
        self._tti=self.encoder.config.model_type.lower() not in ("roberta","xlm-roberta"); self.head=MSDHead(h,n,dropout,inner,n_msd)
    def _pool(self,o,m):
        hs=o.last_hidden_state; cls=hs[:,0]; mean=(hs*m.unsqueeze(-1).float()).sum(1)/m.sum(1,keepdim=True).float().clamp(1); return 0.5*cls+0.5*mean
    def forward(self,ids,mask,tti=None):
        kw={"input_ids":ids,"attention_mask":mask}
        if self._tti and tti is not None: kw["token_type_ids"]=tti
        return self.head(self._pool(self.encoder(**kw),mask))
def uparams(model,e,h,wd):
    nd=["bias","LayerNorm.weight","LayerNorm.bias"]; g=[]
    for grp,lr in [(model.encoder,e),(model.head,h)]:
        d,nde=[],[]
        for n,p in grp.named_parameters():
            if p.requires_grad: (nde if any(x in n for x in nd) else d).append(p)
        g+=[{"params":d,"lr":lr,"weight_decay":wd},{"params":nde,"lr":lr,"weight_decay":0.0}]
    return g
class FGM:
    def __init__(self,model,eps=1.0,key="word_embeddings"): self.m,self.e,self.k,self.b=model,eps,key,{}
    def attack(self):
        for n,p in self.m.named_parameters():
            if p.requires_grad and self.k in n and p.grad is not None:
                self.b[n]=p.data.clone(); nm=torch.norm(p.grad)
                if nm!=0 and not torch.isnan(nm): p.data.add_(self.e*p.grad/nm)
    def restore(self):
        for n,p in self.m.named_parameters():
            if n in self.b: p.data=self.b[n]
        self.b={}
class EMA:
    def __init__(self,model,decay=0.999): self.d=decay; self.s={n:p.detach().clone() for n,p in model.named_parameters() if p.requires_grad}; self.b={}
    def update(self,model):
        for n,p in model.named_parameters():
            if p.requires_grad: self.s[n].mul_(self.d).add_(p.detach(),alpha=1-self.d)
    def apply_shadow(self,model):
        self.b={n:p.detach().clone() for n,p in model.named_parameters() if p.requires_grad}
        for n,p in model.named_parameters():
            if p.requires_grad: p.data.copy_(self.s[n])
    def restore(self,model):
        for n,p in model.named_parameters():
            if p.requires_grad and n in self.b: p.data.copy_(self.b[n])
        self.b={}
def rdrop_kl(lp,lq):
    pl,ql=F.log_softmax(lp,-1),F.log_softmax(lq,-1); p,q=pl.exp(),ql.exp()
    return 0.5*((p*(pl-ql)).sum(-1)+(q*(ql-pl)).sum(-1)).mean()
@torch.no_grad()
def get_logits(model,loader):
    model.eval(); o=[]
    for b in loader:
        b={k:v.to(device) for k,v in b.items()}; o.append(model(b["input_ids"],b["attention_mask"],b.get("token_type_ids")).cpu())
    return torch.cat(o)
def set_seed(s): random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
print("components ok")

components ok


In [5]:
# ── Train one encoder on a held-out config ──────────────────────────────────
def train_enc(model_key, model_name, seed, train_df, val_df, enc, n_cls, is_script_holdout, save):
    set_seed(seed); os.makedirs(save, exist_ok=True); cfg=CFG
    bs = cfg.get("batch_size_override",{}).get(model_key, cfg["batch_size"])
    if bs != cfg["batch_size"]:
        cfg = {**cfg, "batch_size": bs, "grad_accum_steps": cfg["grad_accum_steps"] * (cfg["batch_size"] // bs)}

    is_script_model = (
        model_key == cfg.get("banglabert_script_key", "banglabert")
        and cfg.get("banglabert_script_only", False)
        and "script" in train_df.columns
    )

    if is_script_model:
        tdf = train_df[train_df["script"].astype(str).str.lower().eq("bangla")].reset_index(drop=True)
        vdf = val_df[val_df["script"].astype(str).str.lower().eq("bangla")].reset_index(drop=True)

        if len(tdf) < 50 or len(vdf) < 20:
            print(f"  skip {model_key}: insufficient Bangla rows in this robustness split")
            return None, None, float("nan")

        print(f"  script-aware robustness: {model_key} train={len(tdf):,}, val={len(vdf):,}")
    else:
        tdf = train_df
        vdf = val_df

    tok=AutoTokenizer.from_pretrained(model_name); coll=Collator(tok)
    lkw=dict(collate_fn=coll,num_workers=cfg["num_workers"],pin_memory=torch.cuda.is_available())
    ds=DS(tdf,tok,cfg["max_length"],enc)
    if cfg["use_balanced_sampler"]:
        tl=DataLoader(ds,batch_size=cfg["batch_size"],sampler=make_sampler(tdf,enc,cfg["sampler_alpha"]),shuffle=False,drop_last=True,**lkw)
    else:
        tl=DataLoader(ds,batch_size=cfg["batch_size"],shuffle=True,drop_last=True,**lkw)
    vl=DataLoader(DS(vdf,tok,cfg["max_length"],enc),batch_size=cfg["eval_batch_size"],shuffle=False,**lkw)
    model=Encoder(model_name,n_cls,cfg["dropout"],cfg["head_hidden_dim"],cfg["n_msd"]).to(device)
    # AFTER
    cw=build_cw(tdf[LABEL],enc,cfg["class_weight_beta"]) if cfg.get("use_cw",True) else None
    focal=FocalLoss(cfg["focal_gamma"],cw,cfg["label_smoothing"]) if cfg.get("use_focal",True) else (lambda lg,t: F.cross_entropy(lg,t,weight=cw,label_smoothing=cfg["label_smoothing"]))
    opt=torch.optim.AdamW(uparams(model,cfg["encoder_lr"],cfg["head_lr"],cfg["weight_decay"]))
    ns=math.ceil(len(tl)/cfg["grad_accum_steps"])*cfg["epochs"]
    sch=get_linear_schedule_with_warmup(opt,int(ns*cfg["warmup_ratio"]),ns)
    scaler=torch.amp.GradScaler("cuda") if (cfg["use_fp16"] and device.type=="cuda") else None
    fgm=FGM(model,cfg["fgm_epsilon"]) if cfg["use_fgm"] else None
    ema=EMA(model,cfg["ema_decay"]) if cfg["use_ema"] else None
    @torch.no_grad()
    def vmacro():
        model.eval(); P,Y=[],[]
        for b in vl:
            b={k:v.to(device) for k,v in b.items()}
            P.extend(model(b["input_ids"],b["attention_mask"],b.get("token_type_ids")).argmax(-1).cpu().numpy()); Y.extend(b["label"].cpu().numpy())
        Y=np.array(Y); m=Y>=0; return f1_score(Y[m],np.array(P)[m],average="macro",zero_division=0)
    best,noimp=-1.0,0
    for ep in range(cfg["epochs"]):
        model.train(); opt.zero_grad(set_to_none=True)
        for step,b in enumerate(tl,1):
            b={k:v.to(device) for k,v in b.items()}; y=b["label"]
            with torch.autocast(device_type=device.type,enabled=scaler is not None):
                l1=model(b["input_ids"],b["attention_mask"],b.get("token_type_ids"))
                if cfg["use_rdrop"]:
                    l2=model(b["input_ids"],b["attention_mask"],b.get("token_type_ids"))
                    loss=0.5*(focal(l1,y)+focal(l2,y))+cfg["rdrop_alpha"]*rdrop_kl(l1,l2)
                else: loss=focal(l1,y)
                loss=loss/cfg["grad_accum_steps"]
            (scaler.scale(loss) if scaler else loss).backward()
            if fgm is not None:
                fgm.attack()
                with torch.autocast(device_type=device.type,enabled=scaler is not None):
                    la=focal(model(b["input_ids"],b["attention_mask"],b.get("token_type_ids")),y)/cfg["grad_accum_steps"]
                (scaler.scale(la) if scaler else la).backward(); fgm.restore()
            if step%cfg["grad_accum_steps"]==0 or step==len(tl):
                if scaler: scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(),cfg["max_grad_norm"])
                (scaler.step(opt),scaler.update()) if scaler else opt.step()
                sch.step(); opt.zero_grad(set_to_none=True)
                if ema is not None: ema.update(model)
        # AFTER
        vm_raw=vmacro(); vm,use_ema=vm_raw,False
        if ema is not None and ep>=2:
            ema.apply_shadow(model); vm_ema=vmacro()
            if vm_ema>=vm_raw: vm,use_ema=vm_ema,True
            ema.restore(model)
        if vm>best:
            best,noimp=vm,0
            if use_ema: ema.apply_shadow(model)
            torch.save(model.state_dict(),f"{save}/best.pt")
            if use_ema: ema.restore(model)
        else: noimp+=1
        if noimp>=cfg["patience"]: break
    model.load_state_dict(torch.load(f"{save}/best.pt",map_location=device,weights_only=True))
    torch.save(get_logits(model,vl),f"{save}/val_logits.pt")
    return model, tok, best
print("train_enc ok")

train_enc ok


In [6]:
# ── Run robustness over all held-out configs ────────────────────────────────
def metrics(yt,yp,pr,nc):
    rec={"macro_f1":round(float(f1_score(yt,yp,average="macro",zero_division=0)),4),
         "weighted_f1":round(float(f1_score(yt,yp,average="weighted",zero_division=0)),4),
         "accuracy":round(float(accuracy_score(yt,yp)),4),
         "mcc":round(float(matthews_corrcoef(yt,yp)),4)}
    try: rec["macro_auroc"]=round(float(roc_auc_score(yt,pr,multi_class="ovr",average="macro",labels=list(range(nc)))),4)
    except Exception: rec["macro_auroc"]=float("nan")
    return rec

robust_results=[]
for tag in configs:
    is_scr = tag.startswith("script_holdout")
    tr=pd.read_csv(f"{SPLIT_DIR}/{tag}_train.csv"); va=pd.read_csv(f"{SPLIT_DIR}/{tag}_val.csv"); te=pd.read_csv(f"{SPLIT_DIR}/{tag}_test.csv")
    # HARD leakage guard
    seen=set(tr["uid"])|set(va["uid"]); leaked=len(seen & set(te["uid"]))
    assert leaked==0, f"LEAK in {tag}: {leaked}"
    classes=sorted(pd.concat([tr[LABEL],va[LABEL],te[LABEL]]).unique()); enc={c:i for i,c in enumerate(classes)}; nc=len(classes)
    if DEBUG:
        tr=pd.concat([g.sample(min(len(g),DEBUG_SAMPLES//nc),random_state=42) for _,g in tr.groupby(LABEL)]).reset_index(drop=True)
        va=va.sample(min(len(va),300),random_state=42); te=te.sample(min(len(te),300),random_state=42)
    print(f"\n=== {tag} | classes={nc} | train={len(tr):,} val={len(va):,} test={len(te):,} | leak=0 ✅ ===")
    y_val=va[LABEL].map(enc).values; y_test=te[LABEL].map(enc).values
    val_lg, test_lg = {}, {}
    for mk in RUN_MODELS:
        # A Bangla-only specialist can't do a cross-script robustness test → exclude from script holdouts
        if is_scr and mk == CFG.get("banglabert_script_key","banglabert") and CFG.get("banglabert_script_only",False):
            print(f"  ⏭ {mk}: excluded from {tag} (script-isolated specialist)")
            continue
        for sd in SEEDS:
            save=f"{OUT}/{tag}/{mk}_seed{sd}"
            model,tok,bv=train_enc(mk,CFG["models"][mk],sd,tr,va,enc,nc,is_scr,save)
            if model is None:                       # train_enc skipped (insufficient Bangla rows)
                print(f"  ⏭ {mk} seed{sd}: skipped (no Bangla rows in this held-out train)")
                continue
            coll=Collator(tok); lkw=dict(collate_fn=coll,num_workers=0)
            tel=DataLoader(DS(te,tok,CFG["max_length"],enc),batch_size=CFG["eval_batch_size"],shuffle=False,**lkw)
            val_lg[f"{mk}{sd}"]=torch.load(f"{save}/val_logits.pt",map_location="cpu",weights_only=False).float()
            test_lg[f"{mk}{sd}"]=get_logits(model,tel).float()
            print(f"  {mk} seed{sd}: best_val_macroF1={bv:.4f}")
            del model; torch.cuda.empty_cache()
    # weighted ensemble (optimise on heldout val)
    names=list(val_lg.keys())
    def ens(d,w):
        o=None
        for i,n in enumerate(names): o=w[i]*d[n] if o is None else o+w[i]*d[n]
        return o/(np.sum(w)+1e-12)
    def opt_w(d,y,k):
        best,bw=1.0,np.ones(k)/k
        for _ in range(30):
            r=minimize(lambda rw:-f1_score(y,ens(d,np.abs(rw)+1e-9).argmax(-1).numpy(),average="macro",zero_division=0),
                       np.random.dirichlet(np.ones(k)),method="Nelder-Mead",options={"maxiter":1500,"xatol":1e-5})
            if r.fun<best: best,bw=r.fun,np.abs(r.x)+1e-9
        return bw/bw.sum()
    W=opt_w(val_lg,y_val,len(names))
    ens_t=ens(test_lg,W); pr=F.softmax(ens_t,-1).numpy(); yp=ens_t.argmax(-1).numpy()
    m=metrics(y_test,yp,pr,nc); m["config"]=tag; m["held_out"]=tag.replace("source_holdout_","").replace("script_holdout_",""); m["n_test"]=int(len(y_test))
    robust_results.append(m)
    print(f"  → ENSEMBLE on unseen {m['held_out']}: macroF1={m['macro_f1']} wF1={m['weighted_f1']} acc={m['accuracy']} mcc={m['mcc']}")
    json.dump(m, open(f"{OUT}/{tag}/ensemble_metrics.json","w"), indent=2)


=== script_holdout_bangla | classes=5 | train=32,667 val=4,667 test=56,989 | leak=0 ✅ ===


tokenizer_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/874 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


KeyboardInterrupt: 

In [ ]:
# ── Robustness summary ──────────────────────────────────────────────────────
if robust_results:
    rdf=pd.DataFrame(robust_results)[["config","held_out","n_test","macro_f1","weighted_f1","accuracy","mcc","macro_auroc"]]
    print(rdf.to_string(index=False))
    rdf.to_csv(f"{OUT}/robustness_summary.csv", index=False)
    print(f"\nMean across held-outs: macroF1={rdf['macro_f1'].mean():.4f}  wF1={rdf['weighted_f1'].mean():.4f}")
    print(f"✅ saved {OUT}/robustness_summary.csv")